# LSSTCam — HEALPix Visit Count Sky Maps

- **Creation date:** 2026-03-09  
- **Author:** Sylvie Dagoret-Campagne  
- **Based on:** `2026-03-09_ConsDB_LSSTCam.ipynb`

This notebook queries ConsDB to retrieve LSSTCam visit data (same query as the parent notebook)  
and converts the visit pointing coordinates into **HEALPix visit-count sky maps** using `healpy`.

Outputs produced:
1. One all-band Mollweide sky map.
2. Six per-band sky maps (*u, g, r, i, z, y*).
3. A compact summary panel.
4. A statistics table and bar charts comparing visit counts and sky coverage per band.

**Key design choice for overlays:** `hp.projplot`, `hp.projscatter`, and `hp.projtext`  
are **projection-aware** healpy functions.  With `lonlat=True` they accept plain  
ICRS (RA, Dec) in degrees — no manual RA-wrapping is required.

> Survey status: https://survey-strategy.lsst.io/progress/sv_status/sv_20250930.html


- Check status of the survey : https://survey-strategy.lsst.io/progress/sv_status/sv_20250930.html

## 1. Imports

Standard scientific stack plus `healpy` for HEALPix sky pixelisation and visualisation.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp

from astropy.table import join
from astropy.coordinates import SkyCoord
import astropy.units as u

from lsst.summit.utils import ConsDbClient

#%matplotlib widget
%matplotlib inline

print(f'healpy  version : {hp.__version__}')
print(f'numpy   version : {np.__version__}')

## 2. Configuration

Define the HEALPix resolution (`NSIDE`), band colours, and DDF coordinates.


In [ ]:
# ── Instrument ──────────────────────────────────────────────────────────────────────────────
instrument = 'LSSTCam'

# ── HEALPix resolution ────────────────────────────────────────────────────────────
# NSIDE=64  -> pixel size ~ 0.92 deg  (good overview)
# NSIDE=128 -> pixel size ~ 0.46 deg  (finer, still fast)
# NSIDE=256 -> pixel size ~ 0.23 deg  (matches LSSTCam FOV ~3.5 deg diameter)
NSIDE = 64
NPIX  = hp.nside2npix(NSIDE)
print(f'NSIDE={NSIDE}  ->  {NPIX} pixels,  pixel size ~ {np.degrees(hp.nside2resol(NSIDE)):.2f} deg')

# ── Bands ─────────────────────────────────────────────────────────────────────────────────
BANDS = list('ugrizy')
BAND_COLORS = {
    'u': '#9b59b6', 'g': '#2ecc71', 'r': '#e74c3c',
    'i': '#e67e22', 'z': '#3498db', 'y': '#795548',
}

# ── Matplotlib defaults ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize' : (12, 6),
    'axes.labelsize' : 'x-large',
    'axes.titlesize' : 'x-large',
    'xtick.labelsize': 'x-large',
    'ytick.labelsize': 'x-large',
})

# ── Deep Drilling Fields (ICRS degrees) ───────────────────────────────────────────
DDF_NAMES   = ['XMM-LSS', 'COSMOS', 'ECDFS', 'ELAIS-S1', 'EDFS_a', 'EDFS_b']
DDF_RA_DEG  = np.array([35.57, 150.11, 52.98,  9.45, 58.90, 63.60])
DDF_DEC_DEG = np.array([-4.82,   2.23,-28.12,-44.02,-49.32,-47.60])

print('Configuration done.')

## 3. ConsDB connection & data retrieval

Identical queries to the parent notebook `2026-03-09_ConsDB_LSSTCam.ipynb`:  
retrieve all LSSTCam visits since 2025-04-15 and inner-join `visit1` with itself.


In [ ]:
os.environ['no_proxy'] += ',.consdb'
url    = 'http://consdb-pq.consdb:8080/consdb'
consdb = ConsDbClient(url)
print('ConsDB client created.')

In [ ]:
# Same queries as in the parent notebook
visits    = consdb.query('SELECT * FROM cdb_lsstcam.visit1 WHERE day_obs >= 20250415')
visits_ql = consdb.query('SELECT * FROM cdb_lsstcam.visit1')

merged_visits = join(visits, visits_ql, keys='visit_id', join_type='inner')

df_visits = visits.to_pandas()
print(f'Visits retrieved: {len(df_visits)}')

In [ ]:
start_id  = df_visits.sort_values(by="visit_id").iloc[0]["visit_id"]
last_id = df_visits.sort_values(by="visit_id").iloc[-1]["visit_id"]

In [ ]:
print(f" first visit {start_id} - last visit {last_id}")

## 4. Data cleaning

Remove non-science entries (engineering filters, pinhole mask) and drop visits  
with missing pointing coordinates.


In [ ]:
# Remove engineering / pinhole filters (same as parent notebook)
bad_filters = {'other', 'none', 'other:pinhole'}
df_visits = df_visits[~df_visits['physical_filter'].isin(bad_filters)]

# Remove rows with missing pointing coordinates
df_visits = df_visits.dropna(subset=['s_ra', 's_dec']).reset_index(drop=True)

print(f'Visits after cleaning : {len(df_visits)}')
print(f'Bands present         : {sorted(df_visits["band"].dropna().unique())}')
print(f'RA  range (deg)       : [{df_visits["s_ra"].min():.2f}, {df_visits["s_ra"].max():.2f}]')
print(f'Dec range (deg)       : [{df_visits["s_dec"].min():.2f}, {df_visits["s_dec"].max():.2f}]')

## 5. Helper functions

- `visits_to_healpix_map()` converts (RA, Dec) arrays into a HEALPix visit-count map.
- `galactic_plane_icrs()` traces the Galactic plane in ICRS coordinates.
- `plot_healpix_map()` renders a Mollweide map with all standard overlays.

All overlays use `hp.projplot` / `hp.projscatter` with `lonlat=True` —  
projection-aware, so no manual RA-wrapping is needed.


In [ ]:
def visits_to_healpix_map(ra_deg, dec_deg, nside=NSIDE):
    """
    Build a HEALPix visit-count map from pointing coordinates.

    Parameters
    ----------
    ra_deg, dec_deg : array-like, degrees (ICRS)
    nside : int -- HEALPix NSIDE parameter

    Returns
    -------
    hpmap : np.ndarray shape (hp.nside2npix(nside),)
        Visit count per pixel; empty pixels set to hp.UNSEEN.
    """
    ra  = np.asarray(ra_deg,  dtype=float)
    dec = np.asarray(dec_deg, dtype=float)

    # HEALPix colatitude: theta = 90 - dec  (degrees -> radians)
    theta = np.radians(90.0 - dec)
    phi   = np.radians(ra)

    ipix  = hp.ang2pix(nside, theta, phi)
    hpmap = np.zeros(hp.nside2npix(nside), dtype=float)
    np.add.at(hpmap, ipix, 1)
    hpmap[hpmap == 0] = hp.UNSEEN
    return hpmap


def galactic_plane_icrs():
    """
    Return (ra_deg, dec_deg) of the Galactic plane in ICRS,
    sorted by RA to avoid spurious connecting lines at the 0/360 deg wrap.
    """
    gal_l = np.linspace(0., 360., 1440)
    gal_b = np.zeros(1440)
    gp    = SkyCoord(l=gal_l * u.deg, b=gal_b * u.deg, frame='galactic').icrs
    idx   = np.argsort(gp.ra.deg)
    return gp.ra.deg[idx], gp.dec.deg[idx]


# Galactic plane in plain ICRS degrees -- hp.projplot is projection-aware,
# so no manual Mollweide RA-wrapping is needed.
GP_RA_DEG, GP_DEC_DEG = galactic_plane_icrs()

print('Helper functions defined.')

In [ ]:
def plot_healpix_map(hpmap, title, cmap='YlOrRd', unit='N visits',
                     nside=NSIDE, show_gp=True, show_ddf=True):
    """
    Display a HEALPix visit-count map in Mollweide projection.

    Overlays use hp.projplot / hp.projscatter with lonlat=True:
    they accept (RA_deg, Dec_deg) in ICRS and are fully projection-aware,
    so the Galactic plane and DDFs appear at the correct position
    regardless of the flip/coord settings of hp.mollview.
    """
    hp.mollview(
        hpmap,
        title=title,
        cmap=cmap,
        unit=unit,
        norm=None,
        bgcolor='white',
        badcolor='lightgrey',
        flip='astro',   # RA increases to the left (standard astro convention)
        coord='C',      # Celestial ICRS
    )
    hp.graticule(dpar=30, dmer=30, alpha=0.4, lw=0.5)

    if show_gp:
        # lonlat=True: first arg = RA (deg), second = Dec (deg)
        hp.projplot(GP_RA_DEG, GP_DEC_DEG,
                    ',', ms=1, color='steelblue', label='Galactic Plane',
                    lonlat=True, coord='C')

    if show_ddf:
        hp.projscatter(DDF_RA_DEG, DDF_DEC_DEG,
                       marker='+', s=150, c='black', edgecolors='black',lw=0.5,
                       zorder=10, lonlat=True, coord='C')
        for i, name in enumerate(DDF_NAMES):
            hp.projtext(DDF_RA_DEG[i] + 1.5, DDF_DEC_DEG[i] + 1.5,
                        name, fontsize=10, color='red',
                        fontweight='bold', lonlat=True, coord='C')

    if show_gp:
        plt.gca().legend(loc='lower right', fontsize=8, markerscale=2)

    #plt.tight_layout()
    plt.show()


print('Plotting function defined.')

## 6. HEALPix map — all visits combined

Build and display the combined visit-count map using all bands.


In [ ]:
hpmap_all = visits_to_healpix_map(
    df_visits['s_ra'].values,
    df_visits['s_dec'].values,
    nside=NSIDE
)

n_pix_obs = int((hpmap_all != hp.UNSEEN).sum())
n_vis_tot = int(df_visits.shape[0])
print(f'Total visits          : {n_vis_tot}')
print(f'Observed pixels       : {n_pix_obs} / {NPIX}')
print(f'Max visits per pixel  : {int(hpmap_all[hpmap_all != hp.UNSEEN].max())}')

plot_healpix_map(
    hpmap_all,
    title=f'{instrument} — All bands — Visit count map (NSIDE={NSIDE})',
    cmap='YlOrRd',
    unit='N visits',
)

## 7. HEALPix maps — one per band

Build a separate HEALPix map for each LSST band.


In [ ]:
hpmaps_per_band = {}

for band in BANDS:
    df_b = df_visits[df_visits['band'] == band]
    if df_b.empty:
        print(f'  Band {band}: no data, skipping.')
        continue
    hpmaps_per_band[band] = visits_to_healpix_map(
        df_b['s_ra'].values,
        df_b['s_dec'].values,
        nside=NSIDE
    )
    n_pix = int((hpmaps_per_band[band] != hp.UNSEEN).sum())
    n_vis = int(df_b.shape[0])
    vmax  = int(hpmaps_per_band[band][hpmaps_per_band[band] != hp.UNSEEN].max())
    print(f'  Band {band}: {n_vis:5d} visits | {n_pix:4d} pixels | max {vmax} visits/pixel')

In [ ]:
for band, hpmap_b in hpmaps_per_band.items():
    plot_healpix_map(
        hpmap_b,
        title=f'{instrument} — Band {band} — Visit count map (NSIDE={NSIDE})',
        cmap='YlOrRd',
        unit=f'N visits — band {band}',
    )

## 8. Summary panel — compact individual maps

Display each band in a compact 10×5 inch figure for quick side-by-side comparison.


In [ ]:
bands_available = [b for b in BANDS if b in hpmaps_per_band]

if not bands_available:
    print('No band data available.')
else:
    for band in bands_available:
        plt.figure(figsize=(10, 5))
        hp.mollview(
            hpmaps_per_band[band],
            title=f'Band {band}',
            cmap='YlOrRd',
            unit='N visits',
            bgcolor='white',
            badcolor='lightgrey',
            flip='astro',
            coord='C',
        )
        hp.graticule(dpar=30, dmer=30, alpha=0.3, lw=0.5)

        # Galactic plane -- lonlat=True: (RA_deg, Dec_deg)
        hp.projplot(GP_RA_DEG, GP_DEC_DEG,
                    ',', ms=1, color='steelblue',
                    lonlat=True, coord='C')

        # Deep Drilling Fields
        hp.projscatter(DDF_RA_DEG, DDF_DEC_DEG,
                       marker='+', s=150, c='black', edgecolors='black',lw=0.5,
                       zorder=10, lonlat=True, coord='C')
        for i, name in enumerate(DDF_NAMES):
            hp.projtext(DDF_RA_DEG[i] + 1.5, DDF_DEC_DEG[i] + 1.5,
                        name, fontsize=10, color='red',
                        fontweight='bold', lonlat=True, coord='C')

        plt.suptitle(f'{instrument} — Band {band}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 9. Visit count statistics per band

Summary table: visit count, observed pixels, sky area, max / mean / median depth per band.  
Bar charts compare total visits and sky coverage across the 6 bands.


In [ ]:
import pandas as pd

rows = []
valid = hpmap_all[hpmap_all != hp.UNSEEN]
rows.append({
    'band'         : 'ALL',
    'n_visits'     : int(df_visits.shape[0]),
    'n_pixels_obs' : len(valid),
    'sky_area_deg2': len(valid) * hp.nside2pixarea(NSIDE, degrees=True),
    'max_visits'   : int(valid.max()),
    'mean_visits'  : float(valid.mean()),
    'median_visits': float(np.median(valid)),
})

for band in BANDS:
    df_b = df_visits[df_visits['band'] == band]
    if df_b.empty or band not in hpmaps_per_band:
        continue
    valid_b = hpmaps_per_band[band][hpmaps_per_band[band] != hp.UNSEEN]
    rows.append({
        'band'         : band,
        'n_visits'     : int(df_b.shape[0]),
        'n_pixels_obs' : len(valid_b),
        'sky_area_deg2': len(valid_b) * hp.nside2pixarea(NSIDE, degrees=True),
        'max_visits'   : int(valid_b.max()),
        'mean_visits'  : float(valid_b.mean()),
        'median_visits': float(np.median(valid_b)),
    })

df_stats = pd.DataFrame(rows).set_index('band')
print(f'HEALPix pixel area (NSIDE={NSIDE}): {hp.nside2pixarea(NSIDE, degrees=True):.4f} deg^2')
display(df_stats.round(2))

In [ ]:
# Bar charts: total visits and sky area per band
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_bands = df_stats.drop('ALL')
colors   = [BAND_COLORS.get(b, 'grey') for b in df_bands.index]

ax = axes[0]
ax.bar(df_bands.index, df_bands['n_visits'], color=colors, edgecolor='k', linewidth=0.5)
ax.set_xlabel('Band')
ax.set_ylabel('Total visits')
ax.set_title('Total visits per band')
ax.grid(axis='y', alpha=0.4)

ax = axes[1]
ax.bar(df_bands.index, df_bands['sky_area_deg2'], color=colors, edgecolor='k', linewidth=0.5)
ax.set_xlabel('Band')
ax.set_ylabel('Sky area (deg^2)')
ax.set_title(f'Sky area covered per band  (NSIDE={NSIDE})')
ax.grid(axis='y', alpha=0.4)

plt.suptitle(instrument, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()